In [ ]:
# Cell 1: Install dependencies
!pip install mamba-ssm causal-conv1d 2>/dev/null | tail -2
print('Dependencies installed.')

In [ ]:
# Cell 2: Imports and GPU check
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import KFold, train_test_split
import time
import os
import warnings
warnings.filterwarnings('ignore')

try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm loaded successfully')
except ImportError:
    MAMBA_AVAILABLE = False
    print('WARNING: mamba-ssm not available, Mamba experiments will be skipped')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3: Configuration
GROUP_NAME = 'ThompsonF'
GROUP_DISPLAY = "Thompson's Group F"

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data') if os.path.exists('data') else '.'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = f'/content/drive/MyDrive/word_problem_results/{GROUP_NAME}'
except ImportError:
    OUTPUT_DIR = f'./{GROUP_NAME}_results'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data directory: {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Cell 4: Load datasets
df_synthetic = pd.read_csv(os.path.join(DATA_DIR, 'synthetic_100k.csv'))
df_augmented = pd.read_csv(os.path.join(DATA_DIR, 'augmented_100k.csv'))
df_hard = pd.read_csv(os.path.join(DATA_DIR, 'hard_100k.csv'))

datasets = {'synthetic': df_synthetic, 'augmented': df_augmented, 'hard': df_hard}

for name, df in datasets.items():
    n_id = df['label'].sum()
    print(f'{name}: {len(df)} rows, {n_id} identity, {len(df)-n_id} non-identity, '
          f'lengths {df["length"].min()}-{df["length"].max()} (avg {df["length"].mean():.1f})')

print(f'\nSample words:')
print(df_synthetic.head(3))

In [ ]:
# Cell 5: Tokenizer and Dataset
# Thompson F alphabet: {x0, X0, x1, X1, x2, X2, x3, X3, x4, X4}
# Words are space-separated like 'x0 X1 x2'
VOCAB = {'<PAD>': 0, 'x0': 1, 'X0': 2, 'x1': 3, 'X1': 4,
         'x2': 5, 'X2': 6, 'x3': 7, 'X3': 8, 'x4': 9, 'X4': 10}
VOCAB_SIZE = len(VOCAB)

def encode(word):
    return [VOCAB.get(t, 0) for t in word.split()]

print(f'Vocab ({VOCAB_SIZE} tokens): {VOCAB}')
sample = df_synthetic['word'].iloc[0]
print(f'encode("{sample}") = {encode(sample)}')


class WordProblemDataset(Dataset):
    def __init__(self, words, labels):
        self.words = words
        self.labels = labels

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        encoded = encode(self.words[idx])
        return encoded, self.labels[idx], len(encoded)


def collate_fn(batch):
    words, labels, lengths = zip(*batch)
    max_len = max(lengths)
    padded = [w + [0] * (max_len - len(w)) for w in words]
    sorted_indices = sorted(range(len(lengths)), key=lambda i: lengths[i], reverse=True)
    padded_sorted = [padded[i] for i in sorted_indices]
    labels_sorted = [labels[i] for i in sorted_indices]
    lengths_sorted = [lengths[i] for i in sorted_indices]
    return (
        torch.tensor(padded_sorted),
        torch.tensor(labels_sorted, dtype=torch.float),
        torch.tensor(lengths_sorted)
    )

In [ ]:
# Cell 6: Model Definitions

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0,
                           bidirectional=bidirectional)
        lstm_out = hidden_dim * 2 if bidirectional else hidden_dim
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1))

    def forward(self, input_ids, lengths):
        x = self.embedding(input_ids)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hidden, _) = self.lstm(packed)
        if self.bidirectional:
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            hidden = hidden[-1]
        return self.classifier(hidden).squeeze(-1)


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4,
                 num_layers=2, dropout=0.3, ff_dim=256, max_len=200):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoding = nn.Embedding(max_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, 1))

    def forward(self, input_ids, lengths):
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, -1)
        x = self.embedding(input_ids) + self.pos_encoding(positions)
        mask = (input_ids == 0)
        x = self.transformer(x, src_key_padding_mask=mask)
        mask_exp = (~mask).unsqueeze(-1).float()
        pooled = (x * mask_exp).sum(dim=1) / mask_exp.sum(dim=1).clamp(min=1)
        return self.classifier(pooled).squeeze(-1)


class MambaClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_layers=2,
                 d_state=16, d_conv=4, expand=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(Mamba(d_model=embed_dim, d_state=d_state,
                                    d_conv=d_conv, expand=expand))
            self.norms.append(nn.LayerNorm(embed_dim))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, 1))

    def forward(self, input_ids, lengths):
        x = self.embedding(input_ids)
        for layer, norm in zip(self.layers, self.norms):
            x = x + self.dropout(layer(norm(x)))
        mask = (input_ids != 0).unsqueeze(-1).float()
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.classifier(pooled).squeeze(-1)


def make_model(model_type):
    if model_type == 'LSTM':
        return LSTMClassifier(VOCAB_SIZE).to(device)
    elif model_type == 'Transformer':
        return TransformerClassifier(VOCAB_SIZE).to(device)
    elif model_type == 'Mamba':
        return MambaClassifier(VOCAB_SIZE).to(device)
    else:
        raise ValueError(f'Unknown model type: {model_type}')

for mt in ['LSTM', 'Transformer'] + (['Mamba'] if MAMBA_AVAILABLE else []):
    m = make_model(mt)
    p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{mt}: {p:,} parameters')

In [ ]:
# Cell 7: Training and Evaluation

def train_model(model, train_loader, val_loader, epochs=30, patience=5, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    best_val_acc = 0
    best_state = None
    wait = 0
    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for input_ids, labels, lengths in train_loader:
            input_ids, labels, lengths = input_ids.to(device), labels.to(device), lengths.to(device)
            optimizer.zero_grad()
            out = model(input_ids, lengths)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            correct += ((torch.sigmoid(out) > 0.5).float() == labels).sum().item()
            total += labels.size(0)
        train_acc = correct / total
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for input_ids, labels, lengths in val_loader:
                input_ids, labels, lengths = input_ids.to(device), labels.to(device), lengths.to(device)
                out = model(input_ids, lengths)
                val_correct += ((torch.sigmoid(out) > 0.5).float() == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state:
        model.load_state_dict(best_state)
        model.to(device)
    return best_val_acc, epoch + 1


def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for input_ids, labels, lengths in loader:
            input_ids, labels, lengths = input_ids.to(device), labels.to(device), lengths.to(device)
            out = model(input_ids, lengths)
            correct += ((torch.sigmoid(out) > 0.5).float() == labels).sum().item()
            total += labels.size(0)
    return correct / total


def test_length_generalization(model, df, train_max_len=50):
    long_df = df[df['length'] > train_max_len]
    if len(long_df) == 0:
        return None, 0
    ds = WordProblemDataset(long_df['word'].tolist(), long_df['label'].tolist())
    loader = DataLoader(ds, batch_size=64, collate_fn=collate_fn)
    return evaluate_model(model, loader), len(long_df)

print('Training and evaluation functions defined.')

In [ ]:
# Cell 8: 5-Fold Cross-Validation Experiment Runner

N_FOLDS = 5
SEEDS = [42, 123, 456]
MODEL_TYPES = ['LSTM', 'Transformer'] + (['Mamba'] if MAMBA_AVAILABLE else [])
DATASET_NAMES = ['synthetic', 'augmented', 'hard']
BATCH_SIZE = 64

total_runs = len(MODEL_TYPES) * len(DATASET_NAMES) * len(SEEDS) * N_FOLDS
print(f'Total training runs: {len(MODEL_TYPES)} architectures x {len(DATASET_NAMES)} datasets '
      f'x {len(SEEDS)} seeds x {N_FOLDS} folds = {total_runs}')
print(f'Estimated time: ~{total_runs * 2}min on A100 (~2min/run)')


def run_cv_experiment(df, model_type, seed, dataset_name):
    results = []
    kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    words = df['word'].values
    labels = df['label'].values

    for fold, (train_idx, test_idx) in enumerate(kfold.split(words)):
        random.seed(seed + fold)
        np.random.seed(seed + fold)
        torch.manual_seed(seed + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed + fold)

        train_words, val_words, train_labels, val_labels = train_test_split(
            words[train_idx], labels[train_idx], test_size=0.15, random_state=seed)
        test_words, test_labels = words[test_idx], labels[test_idx]

        train_ds = WordProblemDataset(train_words.tolist(), train_labels.tolist())
        val_ds = WordProblemDataset(val_words.tolist(), val_labels.tolist())
        test_ds = WordProblemDataset(test_words.tolist(), test_labels.tolist())

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate_fn)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, collate_fn=collate_fn)

        model = make_model(model_type)
        t0 = time.time()
        best_val, n_epochs = train_model(model, train_loader, val_loader)
        train_time = time.time() - t0

        test_acc = evaluate_model(model, test_loader)
        len_gen_acc, len_gen_n = test_length_generalization(model, df)

        print(f'  [{model_type}] {dataset_name} seed={seed} fold={fold+1}/{N_FOLDS}: '
              f'test_acc={test_acc:.4f} val_acc={best_val:.4f} ({n_epochs}ep, {train_time:.0f}s)'
              f'{f" len_gen={len_gen_acc:.4f}" if len_gen_acc else ""}')

        results.append({
            'group': GROUP_NAME, 'model': model_type, 'dataset': dataset_name,
            'seed': seed, 'fold': fold + 1, 'test_acc': test_acc,
            'val_acc': best_val, 'train_time': train_time,
            'epochs': n_epochs, 'len_gen_acc': len_gen_acc,
            'len_gen_count': len_gen_n
        })

        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return results

In [ ]:
# Cell 9: Run All Experiments

all_results = []
run_count = 0

for dataset_name in DATASET_NAMES:
    df = datasets[dataset_name]
    for model_type in MODEL_TYPES:
        for seed in SEEDS:
            print(f'\n{"="*60}')
            print(f'{model_type} | {dataset_name} | seed={seed}')
            print(f'{"="*60}')
            fold_results = run_cv_experiment(df, model_type, seed, dataset_name)
            all_results.extend(fold_results)
            run_count += len(fold_results)

            pd.DataFrame(all_results).to_csv(
                os.path.join(OUTPUT_DIR, f'{GROUP_NAME}_results_partial.csv'), index=False)
            print(f'  Progress: {run_count}/{total_runs} runs complete')

print(f'\n{"="*60}')
print(f'ALL {run_count} RUNS COMPLETE')
print(f'{"="*60}')

In [ ]:
# Cell 10: Save Results and Summary

results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(OUTPUT_DIR, f'{GROUP_NAME}_results.csv'), index=False)
print(f'Full results saved to {OUTPUT_DIR}/{GROUP_NAME}_results.csv')
print(f'Total runs: {len(results_df)}')
print()

summary = results_df.groupby(['model', 'dataset']).agg(
    test_acc_mean=('test_acc', 'mean'),
    test_acc_std=('test_acc', 'std'),
    val_acc_mean=('val_acc', 'mean'),
    train_time_mean=('train_time', 'mean'),
    len_gen_acc_mean=('len_gen_acc', 'mean'),
).round(4)

summary['test_acc'] = summary.apply(
    lambda r: f"{r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}", axis=1)

summary.to_csv(os.path.join(OUTPUT_DIR, f'{GROUP_NAME}_summary.csv'))
print(f'Summary saved to {OUTPUT_DIR}/{GROUP_NAME}_summary.csv')
print()
print(summary[['test_acc', 'val_acc_mean', 'train_time_mean', 'len_gen_acc_mean']].to_string())

In [ ]:
# Cell 11: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pivot = results_df.pivot_table(values='test_acc', index='dataset', columns='model', aggfunc='mean')
pivot.plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title(f'{GROUP_DISPLAY}: Test Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.5, 1.02)
axes[0].legend(title='Model')

pivot_time = results_df.pivot_table(values='train_time', index='dataset', columns='model', aggfunc='mean')
pivot_time.plot(kind='bar', ax=axes[1], rot=0)
axes[1].set_title(f'{GROUP_DISPLAY}: Training Time')
axes[1].set_ylabel('Seconds')
axes[1].legend(title='Model')

lg = results_df.dropna(subset=['len_gen_acc'])
if len(lg) > 0:
    pivot_lg = lg.pivot_table(values='len_gen_acc', index='dataset', columns='model', aggfunc='mean')
    pivot_lg.plot(kind='bar', ax=axes[2], rot=0)
    axes[2].set_title(f'{GROUP_DISPLAY}: Length Generalization')
    axes[2].set_ylabel('Accuracy (len > 50)')
    axes[2].set_ylim(0.0, 1.02)
    axes[2].legend(title='Model')
else:
    axes[2].text(0.5, 0.5, 'No length gen data', ha='center', va='center')
    axes[2].set_title('Length Generalization')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'{GROUP_NAME}_results.png'), dpi=150)
plt.show()
print(f'Plot saved to {OUTPUT_DIR}/{GROUP_NAME}_results.png')